# Q2.2 Attention on RNN

## **Auxiliar Functions**
### *config.py*:

In [ ]:
from dataclasses import dataclass
from typing import List


@dataclass
class RNAConfig:
    """Global configuration for the RNAcompete Data Pipeline."""

    # Data Path
    DATA_PATH: str = "norm_data.txt" #NOTE: Only change this if you want to use a different path

    # Metadata is an Excel file
    METADATA_PATH: str = "metadata.xlsx" #NOTE: Only change this if you want to use a different path
    METADATA_SHEET: str = "Master List--Plasmid Info"

    # Save Path
    SAVE_DIR: str = "data" #NOTE: Only change this if you want to use a different path

    # Sequence Parameters
    SEQ_MAX_LEN: int = 41
    ALPHABET: str = "ACGUN"

    # Preprocessing
    CLIP_PERCENTILE: float = 99.95
    EPSILON: float = 1e-6  # For numerical stability

    # Split Identifiers
    TRAIN_SPLIT_ID: str = "SetA"
    TEST_SPLIT_ID: str = "SetB"

    VAL_SPLIT_PCT: float = 0.2
    SEED: int = 42 # NOTE: Change this only if you want to test reproducibility


@dataclass
class RNNHyperparamSpace:
    hidden_size: List[int] = (64, 128, 256, 512)
    batch_size: List[int] = (32, 64, 128)
    lr_min: float = 1e-4
    lr_max: float = 1e-2
    dropout_min: float = 0.0
    dropout_max: float = 0.5
    bidirectional_options: List[bool] = (True, False)
    num_epochs: int = 30


@dataclass
class CNNHyperparamSpace:
    kernel_size: List[int] = (3, 5, 7)
    batch_size: List[int] = (32, 64, 128)
    lr_min: float = 1e-4
    lr_max: float = 1e-2
    dropout_min: float = 0.0
    dropout_max: float = 0.5
    conv_params: List[list] = (
        [8, 16],
        [16, 32],
        [16, 32, 64],
        [32, 64, 128]
    )
    fc_params: List[list] = (
        [32],
        [64],
        [128],
        [64, 32],
        [128, 64]
    )
    no_maxpool: List[bool] = (True, False)
    num_epochs: int = 30


### *utils_w_masking.py*:

In [ ]:
import json
import os
import random
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import TensorDataset, DataLoader
from typing import List, Tuple



def configure_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


class RNACompeteLoader:
    def __init__(self, config: RNAConfig):
        """
        Initializes the loader.
        """
        self.cfg = config
        self.meta_df = None
        self.data_df = None
        self.protein_to_id = None

        # Setup Encoding
        self.char_map = {
            'A': np.array([1, 0, 0, 0], dtype=np.float32),
            'C': np.array([0, 1, 0, 0], dtype=np.float32),
            'G': np.array([0, 0, 1, 0], dtype=np.float32),
            'U': np.array([0, 0, 0, 1], dtype=np.float32),
            'N': np.array([0.25, 0.25, 0.25, 0.25], dtype=np.float32)
        }
        self.padding_vec = np.zeros(4, dtype=np.float32)

    def _ensure_data_loaded(self):
        """Helper to load the heavy files only when necessary."""
        if self.data_df is not None:
            return

        # Load Metadata
        print(f"Loading Metadata from {self.cfg.METADATA_PATH}...")
        start_time = time.time()
        try:
            if self.cfg.METADATA_PATH.endswith('.xlsx'):
                # Requires 'openpyxl' installed!
                self.meta_df = pd.read_excel(
                    self.cfg.METADATA_PATH,
                    sheet_name=self.cfg.METADATA_SHEET
                )
            else:
                self.meta_df = pd.read_csv(self.cfg.METADATA_PATH)
        except Exception as e:
            print(f"Error loading metadata: {e}")
            raise e
        print(f"  > Metadata loaded in {time.time() - start_time:.2f} seconds.")

        # Clean column names (strip whitespace)
        self.meta_df.columns = [c.strip() for c in self.meta_df.columns]

        # Create Protein Name -> RNCMPT ID mapping
        self.protein_to_id = pd.Series(
            self.meta_df['Motif_ID'].values,
            index=self.meta_df['Protein_name']
        ).to_dict()

        # Load Data
        print(f"Loading Data from {self.cfg.DATA_PATH}...")
        start_time = time.time()

        # standard RNAcompete is tab-separated
        self.data_df = pd.read_csv(self.cfg.DATA_PATH, sep='\t', low_memory=False)
        print(f"  > Data Matrix loaded in {time.time() - start_time:.2f} seconds.")

        # Clean data columns
        self.data_df.columns = [c.strip() for c in self.data_df.columns]

    def list_proteins(self) -> List[str]:
        """Returns a sorted list of available protein names."""
        self._ensure_data_loaded()
        valid_proteins = []
        matrix_cols = set(self.data_df.columns)

        for name, pid in self.protein_to_id.items():
            if pid in matrix_cols:
                valid_proteins.append(name)

        return sorted(valid_proteins)

    def _encode_sequence(self, seq: str) -> Tuple[np.ndarray, np.ndarray]:
        """One-hot encodes a single RNA sequence."""
        # Handle NaN or non-string sequence entries gracefully
        if not isinstance(seq, str):
            seq = "N" * self.cfg.SEQ_MAX_LEN

        seq = seq.upper()[:self.cfg.SEQ_MAX_LEN]

        encoded = np.zeros((self.cfg.SEQ_MAX_LEN, len(self.padding_vec)), dtype=np.float32)
        mask = np.zeros(self.cfg.SEQ_MAX_LEN, dtype=np.float32)

        for i, base in enumerate(seq):
            encoded[i] = self.char_map.get(base, self.char_map['N'])
            mask[i] = 1.0

        return encoded, mask

    def _preprocess_intensities(self, intensities: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """Applies: Mask NaNs -> Clip -> Log -> Z-score."""
        mask = (~np.isnan(intensities)).astype(np.float32)
        clean_vals = np.nan_to_num(intensities, nan=0.0)

        # Clip
        if np.sum(mask) > 0:
            valid_data = intensities[mask == 1]
            clip_val = np.percentile(valid_data, self.cfg.CLIP_PERCENTILE)
            clean_vals = np.clip(clean_vals, None, clip_val)

        # Log Transform (Shift to positive)
        min_val = np.min(clean_vals)
        shift = 0
        if min_val <= 0:
            shift = abs(min_val) + 1.0
        clean_vals = np.log(clean_vals + shift + self.cfg.EPSILON)

        # Z-Score
        masked_vals = clean_vals[mask == 1]
        if len(masked_vals) > 0:
            mean = np.mean(masked_vals)
            std = np.std(masked_vals) + self.cfg.EPSILON
            clean_vals = (clean_vals - mean) / std

        clean_vals = clean_vals * mask
        return clean_vals, mask

    def get_data(self, protein_name: str, split: str = 'train') -> TensorDataset:
        """
        Main method to get PyTorch Dataset for a specific protein.
        """
        # Check cache first
        os.makedirs(self.cfg.SAVE_DIR, exist_ok=True)
        data_path = os.path.join(self.cfg.SAVE_DIR, f"{protein_name}_{split}_data.pt")

        if os.path.exists(data_path):
            print(f"Found cached data for {protein_name} ({split}). Loading from {data_path}...")
            try:
                tensors = torch.load(data_path, weights_only=True)
                return TensorDataset(*tensors)
            except Exception as e:
                print(f"Cache seems corrupted: {e}. Will reload from scratch.")

        self._ensure_data_loaded()

        if protein_name not in self.protein_to_id:
            raise ValueError(f"Protein '{protein_name}' not found in metadata.")

        rncmpt_id = self.protein_to_id[protein_name]

        if rncmpt_id not in self.data_df.columns:
            raise ValueError(f"ID {rncmpt_id} for {protein_name} missing from data matrix.")

        s_lower = split.lower()

        if s_lower == 'test':
            # Test set is just SetB, nice and simple
            subset = self.data_df[self.data_df['Probe_Set'] == self.cfg.TEST_SPLIT_ID].copy()

        elif s_lower in ['train', 'val']:
            # For train/val, we need to split SetA.
            # We use a fixed seed to ensure grading consistency (everyone gets the same split).
            full_set = self.data_df[self.data_df['Probe_Set'] == self.cfg.TRAIN_SPLIT_ID]

            # Explicitly sort by index to ensure deterministic order before shuffling
            full_set = full_set.sort_index()

            n_samples = len(full_set)
            indices = np.arange(n_samples)

            # Local RandomState prevents messing with global seeds
            rng = np.random.RandomState(self.cfg.SEED)
            rng.shuffle(indices)

            val_size = int(n_samples * self.cfg.VAL_SPLIT_PCT)

            if s_lower == 'val':
                # Validation gets the first chunk
                subset_indices = indices[:val_size]
            else:
                # Train gets the leftovers
                subset_indices = indices[val_size:]

            subset = full_set.iloc[subset_indices].copy()
        else:
            raise ValueError(f"Unknown split '{split}'. Please use 'train', 'val', or 'test'.")

        # Extract Sequences
        raw_seqs = subset['RNA_Seq'].values
        encoded_list = []
        mask_list = []

        for s in raw_seqs:
            encoded, seq_mask = self._encode_sequence(s)
            encoded_list.append(encoded)
            mask_list.append(seq_mask)

        X = np.stack(encoded_list)          # shape: (B, SEQ_MAX_LEN, 4)
        sequence_masks = np.stack(mask_list)  # shape: (B, SEQ_MAX_LEN)

        # Process Intensities
        # Force conversion to numeric (floats), turning any strings/errors into NaN
        raw_intensities = pd.to_numeric(subset[rncmpt_id], errors='coerce').values
        Y, mask = self._preprocess_intensities(raw_intensities)

        # Convert to Tensor
        dataset = TensorDataset(
            torch.FloatTensor(X),                     # (B, 41, 4)
            torch.FloatTensor(sequence_masks),        # (B, 41)
            torch.FloatTensor(Y).unsqueeze(1),        # (B, 1)
            torch.FloatTensor(mask).unsqueeze(1)      # (B, 1)
        )

        # Save for next time
        print(f"Saving processed data to {data_path}...")
        torch.save(dataset.tensors, data_path)

        return dataset


def load_rnacompete_data(protein_name: str, split: str = 'train', config: RNAConfig = None):
    """
    Convenience function to load data for a single protein without manually managing the loader class.
    Note: Instantiates the loader from scratch (loads files).
    For bulk processing, use RNACompeteLoader class directly.
    """
    if config is None:
        config = RNAConfig()

    loader = RNACompeteLoader(config)
    return loader.get_data(protein_name, split)


def load_best_params(json_path):
    """
    Load the best hyperparameters from a JSON file generated by Optuna.

    This function reads a JSON file containing the results of a hyperparameter
    optimization study and extracts the dictionary of best hyperparameters under
    the key 'best_params'. The returned dictionary can then be used to configure
    a model for final training or evaluation.

    Args:
        json_path (str): Path to the JSON file containing the Optuna study results.

    Returns:
        dict: A dictionary mapping hyperparameter names to their optimal values.
    """
    with open(json_path, 'r') as f:
        data = json.load(f)
    return data['best_params']


def reshape_tensor_dataset(ds):
    """
    Convert dataset for 2D CNN input.

    Args:
        ds: Iterable of (x, x_mask, y, mask)
            x -> (seq_len, 4), x_mask -> (seq_len,), y -> (1,), mask -> (1,)

    Returns:
        TensorDataset of (x, x_mask, y, mask) with shapes:
            x -> (N, 1, seq_len, 4), x_mask -> (N, seq_len, 1), y -> (N, 1), mask -> (N, 1)
    """
    x_list, x_mask_list, y_list, mask_list = [], [], [], []

    for x, x_mask, y, mask in ds:
        # Add channel dimension for Conv2D: (1, seq_len, 4)
        x_list.append(x.unsqueeze(0))
        # Expand x_mask to (seq_len, 1) so it can be broadcast over channels
        x_mask_list.append(x_mask.unsqueeze(-1))
        # Keep y and mask as is
        y_list.append(y)
        mask_list.append(mask)

    # Stack all examples along batch dimension
    x_tensor = torch.stack(x_list)          # (N, 1, seq_len, 4)
    x_mask_tensor = torch.stack(x_mask_list) # (N, seq_len, 1)
    y_tensor = torch.stack(y_list)          # (N, 1)
    mask_tensor = torch.stack(mask_list)    # (N, 1)

    return TensorDataset(x_tensor, x_mask_tensor, y_tensor, mask_tensor)


def masked_spearman_correlation(preds, targets, masks):
    """
    Calculates Spearman Rank Correlation on masked data.
    Expects:
        preds: (B, 1)
        targets: (B, 1)
        masks: (B, 1)
    Outputs:
        correlation: scalar
    """
    # Flatten and detach (metrics don't need gradients)
    preds = preds.squeeze().detach()
    targets = targets.squeeze().detach()
    masks = masks.squeeze().bool()

    valid_preds = preds[masks]
    valid_targets = targets[masks]

    if valid_preds.numel() < 2:
        return torch.tensor(0.0)

    # argsort twice gets us the ranks
    pred_ranks = valid_preds.argsort().argsort().float()
    target_ranks = valid_targets.argsort().argsort().float()

    # Pearson on ranks == Spearman
    pred_mean = pred_ranks.mean()
    target_mean = target_ranks.mean()

    pred_var = pred_ranks - pred_mean
    target_var = target_ranks - target_mean

    correlation = (pred_var * target_var).sum() / torch.sqrt((pred_var ** 2).sum() * (target_var ** 2).sum())

    return correlation


def masked_mse_loss(preds, targets, masks):
    """
    Calculates Mean Squared Error, ignoring padded elements.
    Expects:
        preds: (B, 1)
        targets: (B, 1)
        masks: (B, 1)
    Outputs:
        loss: scalar
    """
    # Flatten to 1D
    preds = preds.squeeze()
    targets = targets.squeeze()
    masks = masks.squeeze().bool()

    # Filter out padded values
    masked_preds = preds[masks]
    masked_targets = targets[masks]

    # Handle empty batch case
    if masked_preds.numel() == 0:
        return torch.tensor(0.0, device=preds.device, requires_grad=True)

    # MSE on valid data
    squared_error = (masked_preds - masked_targets) ** 2
    loss = torch.mean(squared_error)

    return loss


def plot(epochs, plottables, filename=None, ylim=None):
    """Plot the plottables over the epochs.

    Plottables is a dictionary mapping labels to lists of values.
    """
    plt.clf()
    plt.xlabel('Epoch')
    for label, plottable in plottables.items():
        plt.plot(epochs, plottable, label=label)
    plt.legend()
    if ylim:
        plt.ylim(ylim)
    if filename:
        plt.savefig(filename, bbox_inches='tight')



### **RNN Model**

In [ ]:
import argparse
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import os
import numpy as np
# from config import RNAConfig
# from utils_w_masking import load_best_params, masked_mse_loss, masked_spearman_correlation, configure_seed, load_rnacompete_data, plot

class RNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, bidirectional, dropout, use_attention=False):
        super().__init__()

        self.bidirectional = bidirectional
        self.use_attention = use_attention
        self.num_directions = 2 if bidirectional else 1
        self.hidden_dim = hidden_size * self.num_directions

        self.rnn = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True,
            bidirectional=bidirectional
        )

        # Attention Layers
        if self.use_attention:
            self.attn_fc = nn.Linear(self.hidden_dim, self.hidden_dim)
            self.attn_score = nn.Linear(self.hidden_dim, 1, bias=False) # scoring vector v

        self.fc = nn.Linear(self.hidden_dim, output_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, x_mask):
        # x: (B, seq_len, alphabet_size)
        rnn_out, _ = self.rnn(x)

        if self.use_attention:
            # Additive attention pooling
            # (B, seq_len, hidden_dim)
            attn_hidden = torch.tanh(self.attn_fc(rnn_out))

            # (B, seq_len, 1) -> (B, seq_len)
            attn_scores = self.attn_score(attn_hidden).squeeze(-1)

            # Mask padding positions
            attn_scores = attn_scores.masked_fill(x_mask == 0, -1e9)

            # Normalize over sequence length (convert to probabilities)
            attn_weights = torch.softmax(attn_scores, dim=1)

            # Weighted sum of hidden states
            pooled = torch.sum(rnn_out * attn_weights.unsqueeze(-1), dim=1)
        else:
            # x_mask: (B, seq_len) → (B, seq_len, 1)
            mask = x_mask.unsqueeze(-1)

            # Zero out padded positions
            masked_rnn_out = rnn_out * mask

            # Sum only valid positions
            sum_hidden = masked_rnn_out.sum(dim=1)

            # Divide by number of valid tokens
            lengths = mask.sum(dim=1).clamp(min=1)

            # Mean pooling
            pooled = sum_hidden / lengths

        pooled = self.dropout(pooled)
        return self.fc(pooled)

def train_epoch_rnn(loader, model, optimizer, device):
    model.train()
    total_loss = 0.0

    for x, x_mask, y, mask in loader:
        x = x.to(device)
        x_mask = x_mask.to(device)
        y = y.to(device)
        mask = mask.to(device)

        optimizer.zero_grad()
        preds = model(x, x_mask)
        loss = masked_mse_loss(preds, y, mask)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

def evaluate_rnn(loader, model, device):
    model.eval()
    total_loss = 0.0
    total_spearman = 0.0
    n_batches = 0

    with torch.no_grad():
        for x, x_mask, y, mask in loader:
            x = x.to(device)
            x_mask = x_mask.to(device)
            y = y.to(device)
            mask = mask.to(device)

            preds = model(x, x_mask)

            loss = masked_mse_loss(preds, y, mask)
            spearman = masked_spearman_correlation(preds, y, mask)

            total_loss += loss.item()
            total_spearman += spearman.item()
            n_batches += 1

    return total_loss / n_batches, total_spearman / n_batches

#### **Run experiment with attention:**

In [ ]:
def run_Q2_AttentionRNN():
  protein_name = "RBFOX1"
  num_epochs = 30
  alphabet_size = 4

  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

  best_params = load_best_params("best_rnn_params.json")

  hidden_size = best_params["hidden_size"]
  batch_size = best_params["batch_size"]
  learning_rate = best_params["lr"]
  dropout = best_params["dropout"]
  bidirectional = best_params["bidirectional"]

  # ---------------- Setting seed ----------------
  configure_seed(RNAConfig.SEED)

  # ---------------- Loading Data ----------------
  train_dataset = load_rnacompete_data(protein_name, split="train")
  val_dataset = load_rnacompete_data(protein_name, split="val")
  test_dataset = load_rnacompete_data(protein_name, split="test")

  train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
  val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
  test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

  # ---------------- Model ----------------
  model = RNN(
      input_size=alphabet_size,
      hidden_size=hidden_size,
      output_size=1,
      bidirectional=bidirectional,
      dropout=dropout,
      use_attention=True
  )

  model = model.to(device)

  optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

  # ---------------- Training ----------------
  train_losses = []
  val_losses = []
  val_spearman = []
  epochs = np.arange(1, num_epochs + 1)
  for epoch in epochs:
      # ---- Train epoch ----
      train_loss = train_epoch_rnn(train_loader, model, optimizer, device)

      # ---- Evaluate validation set (loss + spearman) ----
      val_loss, val_spear = evaluate_rnn(val_loader, model, device)

      # ---- Save metrics ----
      train_losses.append(train_loss)
      val_losses.append(val_loss)
      val_spearman.append(val_spear)

      print(
          f"Epoch {epoch}/{num_epochs} | "
          f"Train MSE: {train_loss:.4f} | "
          f"Val MSE: {val_loss:.4f} | "
          f"Val Spearman: {val_spear:.4f}"
      )

  # ---------------- Saving Model ----------------
  torch.save(model.state_dict(), "rnn_rbfox1.pt")
  print("Model saved as rnn_rbfox1.pt")

  # ---------------- Testing ----------------
  _, test_spearman = evaluate_rnn(test_loader, model, device)
  print(f"Test Spearman: {test_spearman:.4f}")

  # ---------------- Plotting --------------------
  config = f"{batch_size}-{learning_rate}-{hidden_size}-bi{int(bidirectional)}-drop{dropout}"
  os.makedirs("Q2-RNN-results", exist_ok=True)

  plottables = {
      "Train Loss (train set)": train_losses,
      "Validation Loss (val set)": val_losses,
      "Validation Spearman (val set)": val_spearman
  }

  plot(
      epochs,
      plottables,
      filename=f"Q2-RNN-results/RNN-training-plot-{config}.pdf"
  )

  print("Training, evaluation, and plotting finished.")

#### **Results:**

In [ ]:
run_Q2_AttentionRNN()